# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rashmita-21/Flyrank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


### Method choice and why

My task is to identify content items that are likely to be declining.

The target variable is `is_declining_label`, where:
- 1 means the content is declining.
- 0 means the content is not declining.

I selected Logistic Regression as my first model because this is a binary classification problem. Logistic Regression is a suitable starting point because it is relatively simple, interpretable, and provides probabilities that can be used to rank content items.

I will compare this model against my Week 4 rule-based baseline. Both approaches will be evaluated on the same held-out clients and using the same evaluation metrics.

I will exclude `trend_direction` and `trend_pct` because they are used to derive the target label. I will also exclude `content_id` and `client_id` from model features because they are identifiers, not meaningful predictive signals.

I will prioritize an honest and reproducible comparison over choosing a more complex model.


In [2]:
import sys
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np
import sklearn

print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)

# Find the repository or download it if it is not present
repo_candidates = [
    Path.cwd(),
    Path("/content/Flyrank-ML-Internship"),
    Path("/content")
]

REPO_DIR = None

for folder in repo_candidates:
    if (folder / "scripts" / "01_prepare_features.py").exists():
        REPO_DIR = folder
        break

if REPO_DIR is None:
    REPO_DIR = Path("/content/Flyrank-ML-Internship")
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/rashmita-21/Flyrank-ML-Internship.git",
            str(REPO_DIR)
        ],
        check=True
    )

print("\nRepository location:", REPO_DIR)

# Check that the raw dataset exists
RAW_PATH = REPO_DIR / "data" / "raw" / "content_refresh_anonymized.csv"

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Raw dataset not found at {RAW_PATH}. "
        "Check that the repository contains the data/raw CSV file."
    )

# Run the official feature-preparation script
subprocess.run(
    [sys.executable, str(REPO_DIR / "scripts" / "01_prepare_features.py")],
    cwd=REPO_DIR,
    check=True
)

# Load the prepared dataset, which includes the target label
PREPARED_PATH = (
    REPO_DIR / "data" / "processed" / "refresh_feature_vector.csv"
)

df = pd.read_csv(PREPARED_PATH)

# Load the official model feature lists
sys.path.insert(0, str(REPO_DIR / "scripts"))
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

print("\nPrepared dataset shape:", df.shape)

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nDeclining percentage:")
print(round(df["is_declining_label"].mean() * 100, 2), "%")

print("\nNumeric model features:")
print(MODEL_NUMERIC_FEATURES)

print("\nCategorical model features:")
print(MODEL_CATEGORICAL_FEATURES)

# Check required columns
required_columns = (
    ["is_declining_label", "client_id", "content_id"]
    + MODEL_NUMERIC_FEATURES
    + MODEL_CATEGORICAL_FEATURES
)

missing_columns = [
    col for col in required_columns if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("\nSection 1 completed successfully.")

Pandas version: 2.2.3
Scikit-learn version: 1.6.1

Repository location: /content/Flyrank-ML-Internship

Prepared dataset shape: (30000, 52)

Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Declining percentage:
54.21 %

Numeric model features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Categorical model features:
['competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier']

Section 1 completed successfully.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I will use a client-grouped train/test split to evaluate the model fairly.

I selected an 80/20 split using `GroupShuffleSplit`. This ensures that content from the same client does not appear in both the training and testing sets.

I will use a fixed random seed (`42`) so the split can be reproduced.

The test set will remain unseen during model training. I will use this same test set to compare Logistic Regression with my Week 4 baseline.

I will also check the number of clients and the declining-label percentage in each split to understand whether the data is reasonably balanced.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

# Keep rows with a valid target and client ID
model_df = df.dropna(
    subset=["is_declining_label", "client_id"]
).copy()

# Ensure target is integer-valued
model_df["is_declining_label"] = (
    model_df["is_declining_label"].astype(int)
)

# Define feature columns using the repository's official lists
feature_columns = (
    list(MODEL_NUMERIC_FEATURES)
    + list(MODEL_CATEGORICAL_FEATURES)
)

# Remove any accidental duplicates while preserving order
feature_columns = list(dict.fromkeys(feature_columns))

# Confirm prohibited columns are not model features
prohibited_columns = {
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "is_declining_label"
}

feature_columns = [
    col for col in feature_columns
    if col not in prohibited_columns
]

# Grouped 80/20 train-test split by client
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        model_df["is_declining_label"],
        groups=model_df["client_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

# Separate features and target
X_train = train_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_train = train_df["is_declining_label"].copy()
y_test = test_df["is_declining_label"].copy()

# Check that no client appears in both sets
train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

assert train_clients.isdisjoint(test_clients), (
    "Client leakage detected between train and test sets."
)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("\nTraining clients:", len(train_clients))
print("Testing clients:", len(test_clients))

print("\nDeclining rate in training set:",
      round(y_train.mean() * 100, 2), "%")

print("Declining rate in testing set:",
      round(y_test.mean() * 100, 2), "%")

print("\nNumber of model features:", len(feature_columns))
print("\nTrain/test client overlap:", len(train_clients & test_clients))

print("\nGrouped split completed successfully.")

Training rows: 23837
Testing rows: 6163

Training clients: 25
Testing clients: 7

Declining rate in training set: 55.01 %
Declining rate in testing set: 51.1 %

Number of model features: 26

Train/test client overlap: 0

Grouped split completed successfully.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Train + compare vs my baseline

I will train a Logistic Regression model using the training data only.

I will use a preprocessing pipeline to handle missing values, scale numeric features, and encode categorical features. This ensures preprocessing is fitted only on the training data.

I will compare the model with my Week 4 rule-based baseline using Precision@20 and Precision@50. Both approaches will rank the same held-out test content.

The baseline ranks content using the Week 4 rule: older content with sufficient impressions receives a higher priority. The model ranks content by its predicted probability of decline.

I will also report the declining-label base rate in the test set. This helps show whether either ranking method performs better than selecting content at random.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import average_precision_score
import numpy as np
import pandas as pd

# Identify feature types actually present
numeric_features = [
    col for col in MODEL_NUMERIC_FEATURES
    if col in feature_columns
]

categorical_features = [
    col for col in MODEL_CATEGORICAL_FEATURES
    if col in feature_columns
]

# Numeric preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Logistic Regression model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# Train model
model.fit(X_train, y_train)

# Predicted probability of decline
model_scores = model.predict_proba(X_test)[:, 1]

# Week 4 baseline score
# Higher score means higher priority
baseline_scores = (
    (test_df["content_age_days"] >= 180).astype(int)
    + (test_df["impressions_90d"] >= 500).astype(int)
)

# Precision at K
def precision_at_k(y_true, scores, k):
    k = min(k, len(y_true))
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(y_true)[order].mean()

# Compare both methods on the same test rows
results = []

for method_name, scores in [
    ("Week 4 baseline", baseline_scores),
    ("Logistic Regression", model_scores)
]:
    results.append({
        "Method": method_name,
        "Precision@20": precision_at_k(y_test, scores, 20),
        "Precision@50": precision_at_k(y_test, scores, 50),
        "Average Precision": average_precision_score(y_test, scores)
    })

comparison = pd.DataFrame(results)

print("Test-set declining base rate:",
      round(y_test.mean(), 4))

print("\nModel comparison:")
display(comparison.round(4))

Test-set declining base rate: 0.511

Model comparison:


,Method,Precision@20,Precision@50,Average Precision
0,Week 4 baseline,0.4,0.52,0.4980
1,Logistic Regression,0.7,0.72,0.6044


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

I will examine the model's most important features using permutation importance on the held-out test set.

Permutation importance measures how much the model's performance changes when a feature is shuffled. A larger decrease in performance indicates that the feature provides more useful predictive information.

I will also inspect examples where the model ranked content highly but the actual label was not declining, and examples where declining content received a low score.

These errors help identify cases where the model may struggle. I will interpret the results carefully and avoid treating feature importance as proof of causation.

In [5]:
from sklearn.inspection import permutation_importance

# Calculate permutation importance on held-out test data
importance = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": importance.importances_mean,
    "Std": importance.importances_std
}).sort_values("Importance", ascending=False)

print("Top 10 features by permutation importance:")
display(importance_df.head(10).round(4))

# Prepare test predictions for error inspection
error_df = test_df[
    ["content_id", "client_id", "is_declining_label"]
].copy()

error_df["predicted_probability"] = model_scores
error_df["predicted_label"] = (model_scores >= 0.5).astype(int)

# False positives: predicted declining, actually not declining
false_positives = error_df[
    (error_df["predicted_label"] == 1) &
    (error_df["is_declining_label"] == 0)
].sort_values("predicted_probability", ascending=False)

# False negatives: actually declining, predicted not declining
false_negatives = error_df[
    (error_df["predicted_label"] == 0) &
    (error_df["is_declining_label"] == 1)
].sort_values("predicted_probability", ascending=True)

print("\nThree false positives:")
display(false_positives.head(3))

print("\nThree false negatives:")
display(false_negatives.head(3))

# Include useful context for interpreting the errors
context_columns = [
    col for col in [
        "content_age_days",
        "impressions_90d",
        "days_since_last_update",
        "content_type",
        "main_intent"
    ]
    if col in test_df.columns
]

if context_columns:
    print("\nContext for false positives:")
    display(
        test_df.loc[
            false_positives.head(3).index, context_columns
        ]
    )

    print("\nContext for false negatives:")
    display(
        test_df.loc[
            false_negatives.head(3).index, context_columns
        ]
    )

Top 10 features by permutation importance:


,Feature,Importance,Std
5,log_impressions_90d,0.0734,0.0056
6,log_clicks_90d,0.0585,0.0026
14,avg_position,0.0372,0.0016
11,content_age_days,0.0310,0.0029
24,impression_tier,0.0295,0.0040
7,log_sessions_90d,0.0191,0.0008
3,word_count,0.0101,0.0015
16,scroll_rate,0.0093,0.0043
10,days_with_sessions,0.0054,0.0008
18,competition_level,0.0048,0.0009



Three false positives:


,content_id,client_id,is_declining_label,predicted_probability,predicted_label
26614,content_7be5f150dc65,client_f369cb89fc,0,0.945100,1
20736,content_41baf0722ad9,client_8527a891e2,0,0.932088,1
12869,content_5d5653c4eb4f,client_4e07408562,0,0.917577,1



Three false negatives:


,content_id,client_id,is_declining_label,predicted_probability,predicted_label
8407,content_d1e915d03c28,client_4e07408562,1,0.050920,0
29158,content_e18144cbd19d,client_4e07408562,1,0.055430,0
17690,content_c268b1716236,client_e629fa6598,1,0.064807,0



Context for false positives:


,content_age_days,impressions_90d,days_since_last_update,content_type,main_intent
26614,96,290,20,keyword article,informational
20736,275,3115,104,keyword article,informational
12869,421,15101,7,keyword article,informational



Context for false negatives:


,content_age_days,impressions_90d,days_since_last_update,content_type,main_intent
8407,537,2,104,keyword article,informational
29158,545,3,20,keyword article,informational
17690,502,3,20,keyword article,commercial


### Self-check

Before submitting, I confirm:

- [x] Every section above is filled — markdown thinking and the code that backs it.
- [x] The notebook runs top to bottom without errors (Runtime → Run all).
- [x] No client names, URLs, or private queries appear anywhere.
- [x] My claims use careful words: observed, measured, directional, and decision-support.
- [x] The notebook is committed to my repository under `work/notebooks/`.
- [x] I have submitted my repository URL on the assignment card.